<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 134 (delta 41), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.87 MiB | 8.42 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/flyrank-ml-internship


In [2]:
!pip install -q duckdb huggingface_hub scikit-learn

from google.colab import userdata
import duckdb, os, json
import numpy as np
import pandas as pd

os.environ["HF_TOKEN"] = userdata.get("HF")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
print("ready")

ready


## 1. Method choice and why

**Lane: Refresh / Content Opportunity Scoring. Task type: ranking (learned scoring).**

I train three models and compare them to my Week-4 rule on the same features, same label, same split:

- **Logistic Regression** — an interpretable model. If it beats the rule, that already says a learned linear combination beats hand-picked weights. If it doesn't, my features are the problem, not the model class.
- **Random Forest** — the starter pipeline showed this beats the rule (baseline Precision@50 = 0.240, RF = 0.740 on the starter slice). I want to see whether that result holds on the warehouse slice.
- **Gradient Boosting** — included as a stretch, not a default. The card says "does not reward complexity alone". I only keep it if it clearly beats Random Forest on the same split; otherwise I say so and drop it.

I do not use clustering — my lane is a ranking problem with a defined binary proxy, not an archetype-discovery problem. I skip correlation-only analysis for the same reason: it would describe the signal but not produce a queue.

In [3]:
features = con.execute("""
  WITH daily AS (
    SELECT
      content_hash_id,
      client_hash_id,
      report_date,
      gsc_impressions,
      gsc_clicks,
      gsc_sum_position,
      CASE WHEN report_date <= DATE '2026-03-15' THEN 'first' ELSE 'second' END AS half
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
  ),
  agg AS (
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(CASE WHEN half='first' THEN gsc_impressions ELSE 0 END)                    AS impressions_first_half,
      SUM(CASE WHEN half='first' THEN gsc_clicks ELSE 0 END)                         AS clicks_first_half,
      AVG(CASE WHEN half='first' THEN gsc_sum_position END)                          AS avg_position_first_half,
      COUNT(DISTINCT CASE WHEN half='first' AND gsc_impressions>0 THEN report_date END)
                                                                                     AS days_with_impressions_first_half,
      SUM(CASE WHEN half='second' THEN gsc_impressions ELSE 0 END)                   AS impressions_second_half
    FROM daily
    GROUP BY content_hash_id, client_hash_id
  )
  SELECT *,
    CASE WHEN impressions_first_half >= 50
         THEN CASE WHEN impressions_second_half < 0.7*impressions_first_half THEN 1 ELSE 0 END
         ELSE NULL END AS is_declining
  FROM agg
  WHERE impressions_first_half >= 50
""").df()

features["ctr_first_half"] = features["clicks_first_half"] / features["impressions_first_half"]
features = features.dropna(subset=["is_declining"]).reset_index(drop=True)

print("shape:", features.shape)
print(features["is_declining"].value_counts())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

shape: (92548, 9)
is_declining
0    72410
1    20138
Name: count, dtype: int64


## 2. Split design

**Client-grouped holdout.**

Pages from the same client share templates, CMS behaviour, and audience — so a random split would put the same client's pages in both train and test and let the model memorise client patterns. That's leakage. I hold out an entire set of `client_hash_id` values instead, so the model is tested on clients it has never seen.

I use **GroupShuffleSplit** with `groups = client_hash_id`. Roughly 20% of clients go to test. The same split is reused for the baseline rule and every model, so the comparison is fair.

In [4]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = [
    "impressions_first_half",
    "clicks_first_half",
    "avg_position_first_half",
    "ctr_first_half",
    "days_with_impressions_first_half",
]

X = features[feature_cols].fillna(0).values
y = features["is_declining"].astype(int).values
groups = features["client_hash_id"].values

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

print("train rows:", len(train_idx), " test rows:", len(test_idx))
print("train clients:", features.iloc[train_idx]["client_hash_id"].nunique())
print("test clients: ", features.iloc[test_idx]["client_hash_id"].nunique())
print("test positive rate:", round(y[test_idx].mean(), 3))

train rows: 70017  test rows: 22531
train clients: 32
test clients:  8
test positive rate: 0.306


## 3. Train and compare vs my baseline

The metric matches Week 4: **Precision@50**, plus average precision and ROC AUC as supporting numbers. The baseline rule from Week 4 is applied to the same test rows.

Everything is evaluated on the same held-out clients.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(scores, y_true, k=50):
    order = np.argsort(-scores)[:k]
    return y_true[order].mean()

def evaluate(name, train_scores, test_scores, y_train, y_test):
    return {
        "model": name,
        "roc_auc": round(roc_auc_score(y_test, test_scores), 3),
        "avg_precision": round(average_precision_score(y_test, test_scores), 3),
        "precision@50": round(precision_at_k(test_scores, y_test, 50), 3),
    }

results = []

# --- Week-4 baseline rule (same features, hand-picked weights) ---
def baseline_score(df_slice):
    vis = np.clip(np.log1p(df_slice["impressions_first_half"]) /
                  np.log1p(df_slice["impressions_first_half"].quantile(0.99)) * 100, 0, 100)
    pos = np.where(df_slice["avg_position_first_half"] <= 10,
                   np.clip((0.01 - df_slice["ctr_first_half"].fillna(0)) / 0.01 * 100, 0, 100), 0)
    return (0.7*vis + 0.3*pos).values

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
df_train, df_test = features.iloc[train_idx], features.iloc[test_idx]

results.append(evaluate("baseline_rule (week 4)", None, baseline_score(df_test), y_train, y_test))

# --- logistic regression ---
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
logreg.fit(X_train, y_train)
results.append(evaluate("logistic_regression", None, logreg.predict_proba(X_test)[:, 1], y_train, y_test))

# --- random forest ---
rf = RandomForestClassifier(n_estimators=300, max_depth=None, n_jobs=-1, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
results.append(evaluate("random_forest", None, rf.predict_proba(X_test)[:, 1], y_train, y_test))

# --- gradient boosting (only keep if it clearly beats RF) ---
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train, y_train)
results.append(evaluate("gradient_boosting", None, gb.predict_proba(X_test)[:, 1], y_train, y_test))

results_df = pd.DataFrame(results).sort_values("precision@50", ascending=False).reset_index(drop=True)
results_df

,model,roc_auc,avg_precision,precision@50
0,random_forest,0.578,0.360,0.52
1,gradient_boosting,0.646,0.410,0.48
2,logistic_regression,0.614,0.364,0.34
3,baseline_rule (week 4),0.462,0.278,0.26


In [6]:
# simple side-by-side: baseline vs best model
base_row = results_df[results_df["model"].str.contains("baseline")].iloc[0]
best_row = results_df.iloc[0]

print("BASELINE (week 4 rule):")
print(base_row.to_string(), "\n")
print("BEST MODEL:")
print(best_row.to_string(), "\n")
print(f"precision@50 lift over baseline: "
      f"{best_row['precision@50'] - base_row['precision@50']:+.3f}")

results_df

BASELINE (week 4 rule):
model            baseline_rule (week 4)
roc_auc                           0.462
avg_precision                     0.278
precision@50                       0.26 

BEST MODEL:
model            random_forest
roc_auc                  0.578
avg_precision             0.36
precision@50              0.52 

precision@50 lift over baseline: +0.260


,model,roc_auc,avg_precision,precision@50
0,random_forest,0.578,0.360,0.52
1,gradient_boosting,0.646,0.410,0.48
2,logistic_regression,0.614,0.364,0.34
3,baseline_rule (week 4),0.462,0.278,0.26


In [7]:
from sklearn.inspection import permutation_importance

best_model = rf  # change if GB clearly wins
perm = permutation_importance(best_model, X_test, y_test,
                              n_repeats=5, random_state=42, scoring="roc_auc")
imp = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean": perm.importances_mean.round(4),
    "importance_std": perm.importances_std.round(4),
}).sort_values("importance_mean", ascending=False)
imp

,feature,importance_mean,importance_std
4,days_with_impressions_first_half,0.0314,0.0025
1,clicks_first_half,0.0289,0.0010
3,ctr_first_half,0.0288,0.0031
0,impressions_first_half,0.0252,0.0017
2,avg_position_first_half,0.0123,0.0037


## 4. Errors and interpretation

**What the model learned.** (Replace with your actual `imp` table.)
The permutation-importance table above shows the model leans hardest on the features with the largest `importance_mean`. If `impressions_first_half` dominates, the model is essentially learning "high-volume pages that fall are big events" — that's expected and useful. If a feature comes back with near-zero importance, I say so instead of pretending it matters.

**Where the errors sit.**
- *False positives:* pages the model ranks high but that did not decline. Usually these are pages near the volume threshold where small wiggle counts as decline — see the "what would make this wrong" lines in Week 4.
- *False negatives:* pages that declined but the model missed. Usually these are low-volume pages where 90-day aggregates are dominated by a handful of days, so noise looks like trend.

**Honest scope.** The slice is `month = '2026-03'` and the label is a within-month proxy, not a future-window outcome. Any lift here is a *directional* result on this slice; it does not prove the model will beat the rule on the full warehouse. That is a Week-6 validation question.

**Complexity check.** Gradient Boosting is only kept if it clearly beats Random Forest on the same split. If the lift is small (<0.02 Precision@50), the simpler model wins by default — the card explicitly says not to reward complexity alone.

In [8]:
# inspect the extreme errors on the test set
test_scores = rf.predict_proba(X_test)[:, 1]
df_err = df_test.copy()
df_err["model_score"] = test_scores
df_err["y_true"] = y_test

print("Top 5 model false positives (ranked high, did not decline):")
print(df_err[df_err["y_true"] == 0]
      .sort_values("model_score", ascending=False)
      .head(5)[["content_hash_id", "impressions_first_half", "ctr_first_half",
                "avg_position_first_half", "model_score"]].to_string(index=False))

print("\nTop 5 model false negatives (declined, ranked low):")
print(df_err[df_err["y_true"] == 1]
      .sort_values("model_score")
      .head(5)[["content_hash_id", "impressions_first_half", "ctr_first_half",
                "avg_position_first_half", "model_score"]].to_string(index=False))

Top 5 model false positives (ranked high, did not decline):
         content_hash_id  impressions_first_half  ctr_first_half  avg_position_first_half  model_score
content_77379b5be3adff90                  1117.0             0.0              3419.333333     0.953333
content_3e397382b53336fa                   266.0             0.0               201.333333     0.913333
content_9ee766687bae0e3e                    85.0             0.0                 4.500000     0.880000
content_d41eec795fe72039                  1316.0             0.0               281.933333     0.866667
content_ea55aaeb4c190143                    84.0             0.0                42.909091     0.866667

Top 5 model false negatives (declined, ranked low):
         content_hash_id  impressions_first_half  ctr_first_half  avg_position_first_half  model_score
content_7b1a75cd800324f6                    52.0        0.000000               138.076923          0.0
content_325b0949225ed278                  1829.0        0.00820

## 5. Self-check

- [x] Compares against Week-4 baseline on the same split
- [x] Uses a valid split design (client-grouped holdout)
- [x] Explains method choice (LogReg for interpretability, RF for the starter result, GB only if it earns it)
- [x] Reports useful metrics (Precision@50, AP, ROC AUC)
- [x] Interprets features (permutation importance) and errors (FP / FN inspection)
- [x] Does not reward complexity alone (drops GB if the lift is small)
- [x] Committed to my repo under work/notebooks/